# Bubble Sort 氣泡排序動畫

參考附件影片重建的互動式排序示範。全部畫面均由程式即時運算與繪製，不使用預錄影片；上方夾爪會配合每一步比較與交換，自動移動、下降、夾緊、抬起及放開。

執行下方儲存格後，動畫會直接顯示在 Colab；可暫停、重設、重新排列並調整速度。

> 如要更換資料，只需修改程式最上方的 `INITIAL_VALUES`。

In [1]:

# @title Bubble Sort 動畫設定
INITIAL_VALUES = [5, 2, 8, 1, 7, 3, 6, 4]  # @param {type:"raw"}
AUTO_START = True                           # @param {type:"boolean"}

# ===== 建立並直接顯示互動動畫（無須安裝套件） =====
from IPython.display import HTML, display
import base64
import json

if not 4 <= len(INITIAL_VALUES) <= 10:
    raise ValueError("INITIAL_VALUES 請輸入 4～10 個整數。")
if not all(isinstance(v, int) for v in INITIAL_VALUES):
    raise TypeError("INITIAL_VALUES 的每個項目都必須是整數。")

_template = r'''<!doctype html>
<html lang="zh-Hant">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<style>
  :root {
    --bg0:#04101f; --bg1:#08233d; --panel:#071426e8; --line:#83cef4;
    --muted:#7697b4; --text:#eef8ff; --cyan:#79d6ff; --green:#59e4b5;
    --red:#ff647d; --amber:#ffc45c; --bar:#0e2d4b;
    --move-time:368ms; --claw-time:224ms; --bar-time:416ms;
  }
  * { box-sizing:border-box; }
  body { margin:0; padding:18px 8px 24px; background:#eef3f8; color:var(--text);
         font-family:Inter,ui-sans-serif,system-ui,-apple-system,"Segoe UI",sans-serif; }
  .app { width:min(760px,100%); margin:auto; border:1px solid #194466; border-radius:22px;
         overflow:hidden; position:relative; box-shadow:0 20px 55px #06121f40;
         background:
           linear-gradient(#1c527233 1px,transparent 1px),
           linear-gradient(90deg,#1c527233 1px,transparent 1px),
           radial-gradient(circle at 50% 10%,#103d63 0,#071d34 44%,#04101f 100%);
         background-size:32px 32px,32px 32px,auto; }
  .top { padding:25px 26px 8px; text-align:center; }
  h1 { margin:0; letter-spacing:.11em; font-size:clamp(24px,4vw,35px); font-weight:900; }
  .sub { color:#87a6c0; margin-top:3px; letter-spacing:.14em; font-size:12px; }
  .metrics { display:grid; grid-template-columns:1fr 1fr; gap:14px; width:min(440px,85%); margin:15px auto 0; }
  .metric { background:#050e1ce6; border:1px solid #12344e; border-radius:9px; padding:9px 14px;
            display:flex; align-items:center; justify-content:space-between; color:#7892aa;
            font-size:11px; letter-spacing:.1em; font-weight:800; }
  .metric b { font:800 21px ui-monospace,SFMono-Regular,Consolas,monospace; color:var(--cyan); }
  .metric:last-child b { color:var(--amber); }
  .stage { height:325px; margin:4px 28px 0; position:relative; border-top:2px solid #254963;
           border-bottom:2px solid #173c58; overflow:hidden; }
  .rail { position:absolute; top:14px; left:0; right:0; height:2px; background:#315873; }
  .crane { position:absolute; width:180px; height:188px; top:8px; left:0;
           transition:left var(--move-time) cubic-bezier(.2,.8,.2,1); z-index:5; }
  .crane svg { width:100%; height:100%; overflow:visible; filter:drop-shadow(0 0 8px #62cfff33); }
  .crane .beam { stroke:#95d8fb; stroke-width:4.5; fill:none; stroke-linecap:round; stroke-linejoin:round;
                 transition:stroke var(--claw-time),filter var(--claw-time); }
  .crane .metal { fill:#214968; stroke:#93d7f8; stroke-width:2; }
  .crane .lamp { fill:var(--amber); filter:drop-shadow(0 0 6px var(--amber)); }
  .crane .rope { stroke:#91d8fa; stroke-width:3; transform-box:fill-box; transform-origin:center top;
                 transition:transform var(--claw-time) ease,stroke var(--claw-time); }
  .crane .gripper { --drop:29px; --rope-scale:2.7; }
  .crane .claw-head { transition:transform var(--claw-time) cubic-bezier(.2,.8,.2,1); }
  .crane .finger { fill:none; stroke:#9adfff; stroke-width:4; stroke-linecap:round; stroke-linejoin:round;
                   transform-box:fill-box; transition:transform var(--claw-time) ease,stroke var(--claw-time); }
  .crane .finger.left { transform-origin:right top; transform:translateX(-5px) rotate(8deg); }
  .crane .finger.right { transform-origin:left top; transform:translateX(5px) rotate(-8deg); }
  .crane.claw-lower .gripper .rope,.crane.claw-grip .gripper .rope,.crane.claw-release .gripper .rope { transform:scaleY(var(--rope-scale)); }
  .crane.claw-lower .gripper .claw-head,.crane.claw-grip .gripper .claw-head,.crane.claw-release .gripper .claw-head { transform:translateY(var(--drop)); }
  .crane.claw-grip .finger.left,.crane.claw-lift .finger.left { transform:translateX(1px) rotate(-5deg); }
  .crane.claw-grip .finger.right,.crane.claw-lift .finger.right { transform:translateX(-1px) rotate(5deg); }
  .crane.claw-lift .rope { transform:scaleY(1.12); }
  .crane.claw-lift .claw-head { transform:translateY(3px); }
  .crane.claw-grip .finger,.crane.claw-lift .finger { stroke:var(--amber); filter:drop-shadow(0 0 4px var(--amber)); }
  .crane.compare .beam { stroke:var(--cyan); filter:drop-shadow(0 0 4px var(--cyan)); }
  .crane.swap .beam,.crane.swap .rope { stroke:var(--green); filter:drop-shadow(0 0 5px var(--green)); }
  .bars { position:absolute; left:12px; right:12px; bottom:17px; height:175px; }
  .bar { position:absolute; bottom:0; display:flex; align-items:flex-start; justify-content:center;
         border:2px solid #82c8ed; border-radius:8px 8px 5px 5px; color:#f3faff;
         background:linear-gradient(180deg,#153a5c,#0a2139); box-shadow:inset 0 0 16px #65caff0b;
         transition:left var(--bar-time) cubic-bezier(.22,.9,.24,1), transform var(--claw-time) ease,
                    border-color var(--claw-time),background var(--claw-time),box-shadow var(--claw-time); }
  .bar span { margin-top:9px; font:800 18px ui-monospace,SFMono-Regular,Consolas,monospace; }
  .bar.compare { border-color:var(--cyan); background:linear-gradient(180deg,#164d73,#0d2b48);
                 box-shadow:0 0 18px #54cfff55,inset 0 0 15px #60d4ff1f; transform:translateY(-8px); }
  .bar.left-hot { border-color:var(--red); color:#fff; background:linear-gradient(180deg,#66253b,#28192b);
                  box-shadow:0 0 20px #ff60766b; transform:translateY(-6px); z-index:4; }
  .bar.right-hot { border-color:var(--green); color:#fff; background:linear-gradient(180deg,#155e58,#0d3037);
                   box-shadow:0 0 20px #52e7b071; transform:translateY(-6px); z-index:3; }
  .bar.held { transform:translateY(var(--lift,-30px)); box-shadow:0 0 24px #70e8ff78; }
  .bar.sorted { border-color:var(--green); background:linear-gradient(180deg,#124c49,#0b2c35);
                box-shadow:0 0 12px #51e0ac35; }
  .sorted-line { position:absolute; height:4px; border-radius:3px; bottom:7px; right:12px; width:0;
                 background:var(--green); box-shadow:0 0 8px #59e4b5aa; transition:width .35s; }
  .panel { margin:16px 38px 0; background:#040b16f2; border:1px solid #1b3650; border-radius:11px;
           overflow:hidden; box-shadow:0 12px 25px #0007; }
  .panel-head { height:37px; background:#182337; display:flex; align-items:center; gap:7px; padding:0 13px;
                color:#d6e8f5; font:700 13px ui-monospace,SFMono-Regular,Consolas,monospace; }
  .dot { width:9px; height:9px; border-radius:50%; }
  .dot.r{background:#e85d75}.dot.y{background:#dfa94f}.dot.g{background:#59b8a3}
  .filename { margin-left:7px; }
  .code { padding:7px 0 8px; font:500 13px/1.43 ui-monospace,SFMono-Regular,Consolas,monospace;
          color:#9eb1c0; overflow:hidden; }
  .code-line { min-height:18px; display:flex; white-space:pre; border-left:3px solid transparent; }
  .code-line.active { background:linear-gradient(90deg,#6f532d78,transparent 82%); border-left-color:var(--amber); color:#f0f7fc; }
  .ln { flex:0 0 38px; text-align:right; padding-right:11px; color:#4e6578; user-select:none; }
  .kw { color:#74bff0; } .fn { color:#f2bf68; } .cm { color:#657c8e; }
  .vars { display:flex; justify-content:center; gap:clamp(13px,4vw,34px); padding:8px 10px;
          color:#8ba2b6; background:#15263a; font:700 12px ui-monospace,SFMono-Regular,Consolas,monospace; }
  .vars b { color:#d8e9f4; }
  .status { height:31px; margin:7px 34px 0; text-align:center; color:#93adc2;
            font:700 12px/31px ui-monospace,SFMono-Regular,Consolas,monospace; letter-spacing:.06em; }
  .controls { display:flex; flex-wrap:wrap; align-items:center; justify-content:center; gap:9px; padding:12px 20px 18px; }
  button { appearance:none; border:1px solid #2b668a; background:#0d2b45; color:#e9f7ff; border-radius:9px;
           padding:9px 15px; font-weight:800; cursor:pointer; transition:.2s; }
  button:hover { background:#174260; transform:translateY(-1px); } button.primary{border-color:#49bee8;background:#0d5475}
  .speed { display:flex; align-items:center; gap:8px; margin-left:6px; color:#86a7be; font-size:12px; font-weight:700; }
  input[type=range] { width:110px; accent-color:#56cdef; }
  .legend { display:flex; justify-content:center; gap:18px; padding:0 15px 20px; color:#7593aa; font-size:11px; }
  .legend i { display:inline-block; width:9px; height:9px; border-radius:50%; margin-right:5px; }
  @media (max-width:560px) {
    body{padding:0}.app{border-radius:0}.stage{margin-left:12px;margin-right:12px;height:285px}
    .panel{margin-left:12px;margin-right:12px}.code{font-size:10.5px}.top{padding-left:10px;padding-right:10px}
    .bar span{font-size:15px}.crane{width:150px}.controls{padding-left:8px;padding-right:8px}
  }
</style>
</head>
<body>
<main class="app">
  <header class="top">
    <h1>BUBBLE SORT</h1>
    <div class="sub">程式即時執行｜夾爪同步比較與交換</div>
    <div class="metrics">
      <div class="metric"><span>比較 COMPARES</span><b id="compareCount">0</b></div>
      <div class="metric"><span>交換 SWAPS</span><b id="swapCount">0</b></div>
    </div>
  </header>

  <section class="stage" id="stage">
    <div class="rail"></div>
    <div class="crane" id="crane" aria-hidden="true">
      <svg viewBox="0 0 180 188">
        <rect class="metal" x="72" y="4" width="36" height="14" rx="3"/>
        <circle class="lamp" cx="90" cy="11" r="3.5"/>
        <path class="beam" d="M79 19 L45 82 M101 19 L135 82 M79 19 L135 82 M101 19 L45 82 M35 82 H145"/>
        <g class="gripper left-claw">
          <line class="rope" x1="45" y1="82" x2="45" y2="99"/>
          <g class="claw-head">
            <rect class="metal" x="29" y="96" width="32" height="12" rx="3"/>
            <path class="finger left" d="M35 106 V118 L42 124"/>
            <path class="finger right" d="M55 106 V118 L48 124"/>
          </g>
        </g>
        <g class="gripper right-claw">
          <line class="rope" x1="135" y1="82" x2="135" y2="99"/>
          <g class="claw-head">
            <rect class="metal" x="119" y="96" width="32" height="12" rx="3"/>
            <path class="finger left" d="M125 106 V118 L132 124"/>
            <path class="finger right" d="M145 106 V118 L138 124"/>
          </g>
        </g>
      </svg>
    </div>
    <div class="bars" id="bars"></div>
    <div class="sorted-line" id="sortedLine"></div>
  </section>

  <section class="panel">
    <div class="panel-head"><i class="dot r"></i><i class="dot y"></i><i class="dot g"></i><span class="filename">bubble_sort.c</span></div>
    <div class="code" id="code">
      <div class="code-line" data-line="1"><span class="ln">1</span><span><b class="kw">void</b> <b class="fn">bubble_sort</b>(int a[], size_t n)</span></div>
      <div class="code-line" data-line="2"><span class="ln">2</span><span>{</span></div>
      <div class="code-line" data-line="3"><span class="ln">3</span><span>  <b class="kw">for</b> (size_t i = 0; i + 1 &lt; n; i++) {</span></div>
      <div class="code-line" data-line="4"><span class="ln">4</span><span>    bool swapped = false;</span></div>
      <div class="code-line" data-line="5"><span class="ln">5</span><span>    <b class="kw">for</b> (size_t j = 0; j + 1 &lt; n - i; j++) {</span></div>
      <div class="code-line" data-line="6"><span class="ln">6</span><span>      <b class="kw">if</b> (a[j] &gt; a[j + 1]) {</span></div>
      <div class="code-line" data-line="7"><span class="ln">7</span><span>        int tmp = a[j];</span></div>
      <div class="code-line" data-line="8"><span class="ln">8</span><span>        a[j] = a[j + 1];</span></div>
      <div class="code-line" data-line="9"><span class="ln">9</span><span>        a[j + 1] = tmp;</span></div>
      <div class="code-line" data-line="10"><span class="ln">10</span><span>        swapped = true;</span></div>
      <div class="code-line" data-line="11"><span class="ln">11</span><span>      }</span></div>
      <div class="code-line" data-line="12"><span class="ln">12</span><span>    }</span></div>
      <div class="code-line" data-line="13"><span class="ln">13</span><span>    <b class="kw">if</b> (!swapped) break; <b class="cm">// 已排序</b></span></div>
      <div class="code-line" data-line="14"><span class="ln">14</span><span>  }</span></div>
      <div class="code-line" data-line="15"><span class="ln">15</span><span>}</span></div>
    </div>
    <div class="vars"><span>n = <b id="nVar">8</b></span><span>i = <b id="iVar">0</b></span><span>j = <b id="jVar">0</b></span><span>swapped = <b id="swappedVar">false</b></span></div>
  </section>

  <div class="status" id="status">相鄰元素由左至右逐一比較</div>
  <div class="controls">
    <button class="primary" id="startBtn">▶ 開始</button>
    <button id="resetBtn">↺ 重設</button>
    <button id="shuffleBtn">⤨ 重新排列</button>
    <label class="speed">速度 <input id="speed" type="range" min="0.5" max="2.5" step="0.25" value="1.25"><span id="speedText">1.25×</span></label>
  </div>
  <div class="legend"><span><i style="background:var(--cyan)"></i>比較中</span><span><i style="background:var(--red)"></i>較大值</span><span><i style="background:var(--green)"></i>已就位</span></div>
</main>

<script>
(() => {
  const original = __INITIAL_VALUES__;
  let items = [], comparisons = 0, swaps = 0, runToken = 0;
  let running = false, paused = false, completed = false, speed = 1.25;
  let currentI = 0, currentJ = 0, swappedFlag = false;
  const $ = s => document.querySelector(s);
  const bars = $('#bars'), crane = $('#crane'), startBtn = $('#startBtn');
  const maxValue = Math.max(...original.map(Math.abs), 1);

  function makeItems(values) {
    bars.innerHTML = '';
    items = values.map((value, id) => {
      const el = document.createElement('div');
      el.className = 'bar'; el.innerHTML = `<span>${value}</span>`; bars.appendChild(el);
      return { value, id, el };
    });
    requestAnimationFrame(() => positionItems(false));
  }
  function geometry() {
    const n = items.length, gap = n > 9 ? 5 : 9;
    const cell = Math.max(22, (bars.clientWidth - gap * (n - 1)) / n);
    return {gap, cell};
  }
  function xAt(index) { const {gap,cell}=geometry(); return index*(cell+gap); }
  function positionItems(animate=true) {
    const {cell} = geometry();
    items.forEach((item,index) => {
      item.el.style.transitionDuration = animate ? '' : '0s';
      item.el.style.width = `${cell}px`; item.el.style.left = `${xAt(index)}px`;
      item.el.style.height = `${54 + 108 * Math.abs(item.value)/maxValue}px`;
    });
    if (!animate) requestAnimationFrame(() => items.forEach(x => x.el.style.transitionDuration=''));
  }
  function moveCrane(j) {
    const {cell}=geometry();
    const pairCenter = bars.offsetLeft + (xAt(j)+cell/2 + xAt(j+1)+cell/2)/2;
    const craneWidth = crane.getBoundingClientRect().width;
    crane.style.left = `${Math.max(-5, Math.min($('#stage').clientWidth-craneWidth+5, pairCenter-craneWidth/2))}px`;
  }
  function setClawReach(j) {
    const craneRect=crane.getBoundingClientRect();
    const baseTip=craneRect.top + craneRect.height*(124/188);
    const ropeLength=Math.max(10,craneRect.height*(17/188));
    const pair=[items[j],items[j+1]];
    ['.left-claw','.right-claw'].forEach((selector,k) => {
      const barTop=pair[k].el.getBoundingClientRect().top;
      const drop=Math.max(18,Math.min(112,barTop+10-baseTip));
      const unit=crane.querySelector(selector);
      unit.style.setProperty('--drop',`${drop}px`);
      unit.style.setProperty('--rope-scale',String(1+drop/ropeLength));
      pair[k].el.style.setProperty('--lift',`${-(Math.max(22,drop-3))}px`);
    });
  }
  const clawModes=['claw-lower','claw-grip','claw-lift','claw-release'];
  function setClaw(mode='open') {
    crane.classList.remove(...clawModes);
    if (mode!=='open') crane.classList.add(`claw-${mode}`);
  }
  function updateMotion() {
    const root=document.documentElement.style;
    root.setProperty('--move-time',`${460/speed}ms`);
    root.setProperty('--claw-time',`${280/speed}ms`);
    root.setProperty('--bar-time',`${520/speed}ms`);
  }
  function setLine(line) {
    document.querySelectorAll('.code-line').forEach(el => el.classList.toggle('active', +el.dataset.line===line));
  }
  function setStatus(text) { $('#status').textContent = text; }
  function updateHUD() {
    $('#compareCount').textContent=comparisons; $('#swapCount').textContent=swaps;
    $('#nVar').textContent=items.length; $('#iVar').textContent=currentI; $('#jVar').textContent=currentJ;
    $('#swappedVar').textContent=String(swappedFlag);
  }
  function clearTransient() {
    items.forEach(x => x.el.classList.remove('compare','left-hot','right-hot','held'));
    crane.classList.remove('compare','swap');
    setClaw('open');
  }
  function markSorted(from) {
    items.forEach((x,k) => x.el.classList.toggle('sorted', k>=from));
    const {cell,gap}=geometry();
    const count=items.length-from;
    $('#sortedLine').style.width = count ? `${count*cell+(count-1)*gap}px` : '0px';
  }
  function delay(ms, token) {
    return new Promise((resolve,reject) => {
      let remaining=ms, last=performance.now();
      const tick=now => {
        if (token!==runToken) return reject(new Error('cancelled'));
        if (!paused) remaining -= now-last;
        last=now;
        if (remaining<=0) resolve(); else requestAnimationFrame(tick);
      };
      requestAnimationFrame(tick);
    });
  }
  async function highlightSwapLines(token) {
    for (const line of [7,8,9,10]) { setLine(line); await delay(105/speed,token); }
  }
  async function sort(token) {
    running=true; paused=false; completed=false; startBtn.textContent='⏸ 暫停';
    try {
      for (let i=0;i<items.length-1;i++) {
        currentI=i; swappedFlag=false; setLine(4); updateHUD();
        setStatus(`第 ${i+1} 輪：將目前最大值向右移動`);
        await delay(360/speed,token);
        for (let j=0;j<items.length-1-i;j++) {
          currentJ=j; setLine(5); clearTransient(); setClawReach(j); moveCrane(j); crane.classList.add('compare');
          items[j].el.classList.add('compare'); items[j+1].el.classList.add('compare'); updateHUD();
          setStatus(`夾爪移動至 ${items[j].value} 與 ${items[j+1].value} 的上方`);
          await delay(360/speed,token);
          setClaw('lower'); setStatus('夾爪下降並對準兩個相鄰元素');
          await delay(230/speed,token);
          setClaw('grip'); setStatus('夾爪夾緊，程式執行大小比較');
          await delay(200/speed,token);
          comparisons++; setLine(6); updateHUD();
          const a=items[j].value, b=items[j+1].value;
          if (a>b) {
            swappedFlag=true; swaps++; crane.classList.replace('compare','swap');
            items[j].el.classList.remove('compare'); items[j+1].el.classList.remove('compare');
            items[j].el.classList.add('left-hot'); items[j+1].el.classList.add('right-hot');
            setStatus(`${a} > ${b}，交換兩個相鄰元素`); updateHUD();
            await highlightSwapLines(token);
            items[j].el.classList.add('held'); items[j+1].el.classList.add('held');
            setClaw('lift'); setStatus(`夾爪抬起 ${a} 與 ${b}，並依程式交換位置`);
            await delay(220/speed,token);
            [items[j],items[j+1]]=[items[j+1],items[j]]; positionItems(true);
            await delay(540/speed,token);
            items[j].el.classList.remove('held'); items[j+1].el.classList.remove('held');
            setClaw('release'); setStatus('交換完成，夾爪下降並放開');
            await delay(190/speed,token);
          } else {
            setStatus(`${a} ≤ ${b}，順序正確，不需交換`);
            setClaw('release');
            await delay(260/speed,token);
          }
          clearTransient();
          await delay(100/speed,token);
        }
        markSorted(items.length-1-i); setLine(13); updateHUD();
        await delay(360/speed,token);
        if (!swappedFlag) break;
      }
      markSorted(0); clearTransient(); setLine(13); completed=true;
      setStatus(`排序完成：${items.map(x=>x.value).join('  <  ')}`);
    } catch(e) { if (e.message!=='cancelled') throw e; }
    if (token===runToken) { running=false; paused=false; startBtn.textContent=completed?'✓ 完成':'▶ 開始'; }
  }
  function reset(values=original) {
    runToken++; running=false; paused=false; completed=false; comparisons=0; swaps=0;
    currentI=0; currentJ=0; swappedFlag=false; clearTransient(); makeItems(values.slice());
    setLine(1); markSorted(items.length); updateHUD(); setStatus('相鄰元素由左至右逐一比較');
    startBtn.textContent='▶ 開始';
    setTimeout(()=>{ moveCrane(0); setClawReach(0); setClaw('open'); },80);
  }
  function startOrPause() {
    if (running) { paused=!paused; startBtn.textContent=paused?'▶ 繼續':'⏸ 暫停';
      setStatus(paused?'動畫已暫停':`繼續比較第 ${currentJ+1} 與第 ${currentJ+2} 個元素`); return; }
    if (completed) reset(original);
    const token=++runToken; sort(token);
  }
  startBtn.addEventListener('click',startOrPause);
  $('#resetBtn').addEventListener('click',()=>reset(original));
  $('#shuffleBtn').addEventListener('click',()=>{
    const v=original.slice(); for(let i=v.length-1;i>0;i--){const j=Math.floor(Math.random()*(i+1));[v[i],v[j]]=[v[j],v[i]];} reset(v);
  });
  $('#speed').addEventListener('input',e=>{
    speed=+e.target.value; $('#speedText').textContent=`${speed.toFixed(2)}×`; updateMotion();
  });
  window.addEventListener('resize',()=>{
    positionItems(false);
    if(items.length>1){const j=Math.min(currentJ,items.length-2);moveCrane(j);setClawReach(j);}
  });
  updateMotion();
  reset(original);
  if (__AUTO_START__) setTimeout(startOrPause,850);
})();
</script>
</body>
</html>'''
_page = (_template
         .replace("__INITIAL_VALUES__", json.dumps(INITIAL_VALUES))
         .replace("__AUTO_START__", "true" if AUTO_START else "false"))
_payload = base64.b64encode(_page.encode("utf-8")).decode("ascii")
display(HTML(
    f'<iframe src="data:text/html;base64,{_payload}" '
    'style="width:100%;height:1040px;border:0;border-radius:18px;" '
    'allow="autoplay" loading="eager"></iframe>'
))
